# 📈 informed_flow — optimize

A **daily, cross-sectional** strategy on **alternative data**. Every liquid name is ranked each
day on insider buying (SEC Form 4), analyst rating changes and price-target revisions (Yahoo),
short interest (FINRA) and how recently any of them acted; the book holds the top-ranked names
in `max_positions` slots of one shared cash account, and an optional **risk overlay**
(`risk_method`) decides how much of the book is invested at all. The thesis, the mechanics and
the study history live in [README.md](./README.md).

**This notebook only optimizes.** It reads the **training window only** — the 2025+ holdout
is never loaded here — runs the Optuna study, reads the Pareto front and saves every admissible
front row as a *candidate*. Whether a candidate is worth trading is decided in
[`informed_flow_diagnostics.ipynb`](./informed_flow_diagnostics.ipynb), which compares any set
of candidates on the deep checks, the walk-forward and the holdout.

Flow: **load train → parameters → optimize → read the front → boundary check → the pick vs
buy-and-hold → save candidates.** The usual loop is §3–§5: run, check the boundaries, widen a
bound in `config.yaml` (and bump `optuna.study`), run again.

> Data must be loaded first: run `notebooks/03_altdata_update.ipynb`.

## 1 · Load the training panel

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import pandas as pd
import plotly.graph_objects as go
import yaml

from trading_simple import backtest as bt
from trading_simple import features, jobs, plots, strategies
from trading_simple import optimize as opt
from trading_simple.config import load_settings
from trading_simple.data import load_panel, symbols_with_bars
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
strat = strategies.load("informed_flow")
d = strat.data

syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]), d["bar"],
                         d["start"], d.get("end"), limit=d.get("max_symbols"))
# TRAIN ONLY, on its own window. Never load full history and slice back: a feature COLUMN
# exists only when the query window returned rows for that symbol, so a wider load gives a
# name whose only insider activity is in 2025 an all-zero column over the train years where
# this load leaves it all-NaN — holdout availability leaking into training data.
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
features.attach_features(panel, engine, d["features"], settings=settings)

print(f"{d['universe']}: {len(syms)} symbols, {panel.close.shape[0]} daily bars "
      f"{panel.close.index[0].date()} → {panel.close.index[-1].date()}  (TRAIN)")
for name, frame in panel.features.items():
    active = int((frame.abs() > 0).any().sum())
    print(f"  feature {name:15s}: {active}/{len(syms)} symbols with data")

## 2 · Parameters and search space

Rendered straight from `config.yaml`, so it cannot drift from what runs. A parameter with a
searched range is tuned by §3; the rest ship at their value.

In [ ]:
SPACE = strat.optuna["space"]
print(f"study {strat.optuna['study']} · fill model {strat.fill} · pick floor "
      f"{strat.optuna['pick_min_trades_per_year']} trades/yr\n")
print(f"  {'parameter':18s} {'ships':>12s}   searched")
for k, v in strat.params.items():
    spec = SPACE.get(k)
    if spec is None:
        rng = "—"
    elif spec["type"] == "categorical":
        rng = " | ".join(map(str, spec["choices"]))
    else:
        rng = f"{spec['low']} … {spec['high']}" + (" (log)" if spec.get("log") else "")
    print(f"  {k:18s} {v!s:>12s}   {rng}")

## 3 · Optimize (Optuna, resumable)

`jobs.run_optimize` runs the study in a `tsim optimize` subprocess and streams one line per
trial. It scores **`excess_cagr` (maximized) against `volatility` (minimized)** — beat
equal-weight buy-and-hold, as calmly as possible — so the result is a Pareto front, not a single
winner. The study is journal-backed: **re-running this cell resumes** where it stopped. Every
extra trial is another chance to fit noise, which the diagnostics notebook's deflated Sharpe
charges for.

In [ ]:
N_OPTIMIZE_TRIALS = 2000
jobs.run_optimize("informed_flow", n_trials=N_OPTIMIZE_TRIALS, study=strat.optuna["study"])

## 4 · Read the front — and the pick

Few-trade parameter sets can post a flattering number by luck, so only front rows trading at
least `optuna.pick_min_trades_per_year` are admissible. Among those, `opt.pick_from_front` — the
one picking rule in the repo, which the diagnostics notebook's walk-forward also uses — takes
the balanced best, never a row that loses to buy-and-hold, and collapses tied outcomes first.

The table lists every **distinct admissible outcome**, calmest first: moving down the table
buys excess return with volatility. A row's `trial` number is how you name it as a candidate in
the diagnostics notebook; the pick is in bold.

In [ ]:
study = opt.load_study(strat)
front = opt.pareto_frame(study)
FLOOR = strat.optuna["pick_min_trades_per_year"]
admissible = front[front["trades_per_year"] >= FLOOR]
pool = admissible if len(admissible) else front
best_row = opt.pick_from_front(pool)
PICK = 2268 #int(best_row["trial"])
SPACE_INT = {k for k, spec in SPACE.items() if spec["type"] == "int"}


def as_params(row) -> dict:
    """A front row's searched parameters, cast back to the types signals() expects."""
    out = {}
    for k in SPACE:
        v = row[k]
        if isinstance(v, str):
            out[k] = v
        elif k in SPACE_INT:
            out[k] = int(round(float(v)))
        else:
            out[k] = round(float(v), 10)          # 0.59, not Optuna's 0.5900000000000001
    return out


picked = as_params(best_row)
states = pd.Series([t.state.name for t in study.trials]).value_counts()
print(f"{len(study.trials)} trials ("
      + ", ".join(f"{n} {s.lower()}" for s, n in states.items())
      + f"); front {len(front)} rows, {len(admissible)} admissible (≥ {FLOOR} trades/yr), "
      f"{int(best_row['front_distinct'])} distinct outcomes")
print(f"PICK: trial {PICK}"
      + (f" — tied with {int(best_row['n_tied']) - 1} identical-scoring trial(s)"
         if best_row["n_tied"] > 1 else "")
      + ("" if best_row["beats_benchmark"] else " — NOTHING on the front beats buy-and-hold"))

menu = opt.distinct_front(pool).sort_values("volatility")
cols = [c for c in ("trial", "excess_cagr", "volatility", "info_ratio", "trades_per_year",
                    "risk_method", "max_positions") if c in menu]
display(menu[cols].style.hide(axis="index")
        .format({"excess_cagr": "{:+.1%}", "volatility": "{:.1%}", "info_ratio": "{:+.2f}",
                 "trades_per_year": "{:.0f}"})
        .apply(lambda r: ["font-weight: bold" if r["trial"] == PICK else ""] * len(r), axis=1))
plots.plot_pareto(front, opt.trials_frame(study), picked=PICK).show()

## 5 · Boundary check

If the pick — or at least half of the distinct front — sits on the edge of a searched range, the
real optimum may lie outside it: widen that bound in `config.yaml`, **bump `optuna.study`**, and
run §3 again. A sleeve weight at `0.0` flagged "widen low" is noise (a weight cannot go
negative), and categoricals have no edges. The knobs of a risk method the pick does not use
(`risk_off_exposure` under anything but `cash_switch`, `target_vol` under anything but
`vol_target`) do nothing, so their edge flags are noise too.

In [ ]:
report = opt.boundary_report(picked, SPACE, front=pool)
display(report)
flagged = report[report["action"] != "ok"]
print("no bound is pressing — the search box holds the optimum" if flagged.empty else
      f"widen {dict(flagged['action'])} → edit config.yaml, bump optuna.study, re-run §3")

## 6 · The pick on training data — against buy-and-hold

A sanity look, not a verdict: this is the window the study was fitted on, so every number here
flatters the pick by construction. The deep checks and the untouched holdout are in the
diagnostics notebook. Costs are on, and the dashed line is the mandatory baseline — equal-weight
buy-and-hold of the same names (`strategies/baseline`), paying the same costs.

In [ ]:
INIT_CASH = settings.backtest.init_cash
res = bt.run_strategy(strat, panel, params=picked, settings=settings)
res_base = bt.run_strategy(strategies.load("baseline"), panel, settings=settings)
st, base = bt.portfolio_stats(res), bt.portfolio_stats(res_base)
fees, slip = bt.panel_costs(settings.costs, panel)
st |= bt.active_stats(res, panel, fees=fees, slippage=slip, total_return=st["total_return"],
                      fill=strat.fill)
rows = ["cagr", "volatility", "max_drawdown", "sharpe", "trades_per_year"]
summary = pd.DataFrame({f"pick #{PICK}": [st[k] for k in rows],
                        "buy & hold": [base[k] for k in rows]}, index=rows)
display(summary.style.format("{:+.1%}", subset=pd.IndexSlice[rows[:3], :])
        .format("{:.2f}", subset=pd.IndexSlice[rows[3:], :]))
print(f"excess over buy-and-hold {st['excess_cagr']:+.2%}/yr · information ratio "
      f"{st['info_ratio']:+.2f} · {int(st['n_trades'])} trades")

fig = go.Figure()
fig.add_trace(plots._scatter(
    bt.equity_curve(res_base, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0,
    name="baseline (buy & hold)", line={"color": "#898781", "dash": "dash", "width": 1.5},
    hovertemplate="%{x|%Y-%m-%d}<br>%{y:+.1%}<extra>baseline</extra>"))
fig.add_trace(plots._scatter(
    bt.equity_curve(res, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0,
    name=f"pick #{PICK}", line={"color": plots.SERIES[0], "width": 1.5},
    hovertemplate="%{x|%Y-%m-%d}<br>%{y:+.1%}<extra>pick</extra>"))
fig.update_xaxes(type="date")  # _scatter packs x as epoch ms; this is what keeps it a date axis
plots._base(fig, title=f"{d['universe']} · train — the pick vs equal-weight buy-and-hold",
            height=420)
fig.update_yaxes(tickformat="+.0%", title_text="cumulative return")
fig.show()
display(bt.symbol_stats(res).head(10).round(3))

## 7 · Save the candidates for the diagnostics notebook

Every distinct admissible front row is written to `studies/<study>.candidates.yaml` with its
parameters, its training numbers and which one the picking rule chose. The file is a snapshot:
re-running §3 grows the journal and can move the front, but the diagnostics notebook keeps
comparing exactly what was saved here until this cell runs again. It is git-ignored, like the
journal it came from.

**Next:** open `informed_flow_diagnostics.ipynb`, set `CANDIDATES` (e.g. `["pick", 1191]`) and
Run All.

In [ ]:
path = opt.study_path(strat, strat.optuna["study"]).with_suffix(".candidates.yaml")
METRICS = ("excess_cagr", "volatility", "info_ratio", "tracking_error", "cagr", "max_drawdown",
           "trades_per_year", "years")
snapshot = {
    "study": strat.optuna["study"],
    "saved": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
    "pick_min_trades_per_year": FLOOR,
    "pick": PICK,
    "front": [{"trial": int(r["trial"]),
               **{m: round(float(r[m]), 6) for m in METRICS if m in r},
               "params": as_params(r)}
              for _, r in menu.iterrows()],
}
path.write_text(yaml.safe_dump(snapshot, sort_keys=False))
print(f"saved {len(snapshot['front'])} candidates (pick #{PICK}) -> {path}")
print("trials:", [c["trial"] for c in snapshot["front"]])